# Phase 3 Validation: Scoring & Backtesting

**Purpose** - understand what the scoring engine produces and whether its signals have any edge *before* we build AI/strategy layers on top.

**How to read this notebook** (top to bottom):
1. **Data** - what we are testing on (DB if reachable, otherwise a synthetic demo so the notebook always runs).
2. **Scoring validation** - score/signal distributions and *forward returns by signal*. If BUY does not beat AVOID here, nothing downstream can work.
3. **Backtest** - long/flat strategy, next-bar-open fills, fees + slippage, vs buy & hold.
4. **Equity, drawdown, trades** - the story behind the numbers.
5. **Sensitivity** - how fragile the result is to fees, stops and thresholds.
6. **All pairs + persisted runs + live pipeline health**.
7. **Go / no-go checklist** for moving to the AI-intelligence phase.

> Synthetic data is a random walk: expect *no* edge there. It only proves the machinery works.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1] / 'src'))
warnings.filterwarnings('ignore')

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from dataclasses import asdict

from cryptoquant.backtesting.engine import BacktestConfig, run_backtest
from cryptoquant.backtesting.synthetic import make_synthetic_bars
from cryptoquant.strategies.signal_strategy import SignalStrategy, ScoreStrategy

pd.options.display.float_format = '{:,.4f}'.format
plt.rcParams['figure.figsize'] = (12, 4)

# ---- Parameters -----------------------------------------------------------
SYMBOL = 'BTC-USD'
START = None            # e.g. '2025-01-01'
END = None
FORWARD_HOURS = 24      # horizon for forward-return validation
CONFIG = BacktestConfig(initial_capital=10_000, fee_bps=40, slippage_bps=5, stop_atr_mult=None)

In [ ]:
USE_DB = False
try:
    from cryptoquant.database.session import get_session
    from cryptoquant.backtesting.repository import load_bars, get_tracked_pairs
    session = get_session()
    pairs = dict((s, i) for i, s in get_tracked_pairs(session))
    bars = load_bars(session, pairs[SYMBOL],
                     pd.Timestamp(START) if START else None, pd.Timestamp(END) if END else None)
    USE_DB = len(bars) > 0
except Exception as exc:
    print(f'DB unavailable ({type(exc).__name__}); falling back to synthetic data.')

if not USE_DB:
    SYMBOL = 'SYNTHETIC'
    bars = make_synthetic_bars(n=4000)

df = pd.DataFrame([asdict(b) for b in bars]).set_index('timestamp')
print(f'Source: {"Azure SQL" if USE_DB else "SYNTHETIC"} | {SYMBOL} | {len(df):,} bars | {df.index.min()} -> {df.index.max()}')
print(f'Scored bars: {df.signal.notna().sum():,} ({df.signal.notna().mean():.0%})')
df.tail(3)

## 1. Scoring validation
If the scores were NULL here, run `python scripts/rescore_analysis.py` first.

In [ ]:
scored = df.dropna(subset=['signal']).copy()
fig, ax = plt.subplots(1, 2)
scored.score.hist(bins=40, ax=ax[0]); ax[0].set_title('Technical score distribution (0-100)')
ax[0].axvline(35, c='r', ls='--'); ax[0].axvline(65, c='g', ls='--')
scored.signal.value_counts().reindex(['BUY', 'HOLD', 'AVOID']).plot.bar(ax=ax[1], color=['g', 'gray', 'r'])
ax[1].set_title('Signal counts'); plt.tight_layout(); plt.show()
scored.signal.value_counts(normalize=True).rename('share').to_frame()

### Forward returns by signal (the key validation)
For every scored bar: return from the **next open** to the open `FORWARD_HOURS` later (the same no-look-ahead fill the backtester uses).
We want **BUY > HOLD > AVOID**. If the ordering is not monotonic, the score has no predictive value on this data.

In [ ]:
fwd = df.open.shift(-(FORWARD_HOURS + 1)) / df.open.shift(-1) - 1
scored = df.assign(fwd=fwd).dropna(subset=['signal', 'fwd'])
by_sig = scored.groupby('signal').fwd.agg(['count', 'mean', 'median', lambda s: (s > 0).mean()])
by_sig.columns = ['bars', 'mean_fwd', 'median_fwd', 'hit_rate']
by_sig = by_sig.reindex(['BUY', 'HOLD', 'AVOID'])
display(by_sig)

scored['decile'] = pd.qcut(scored.score, 10, labels=False, duplicates='drop')
dec = scored.groupby('decile').fwd.mean()
dec.plot.bar(title=f'Mean {FORWARD_HOURS}h forward return by score decile (0 = lowest score)')
plt.axhline(0, c='k', lw=.5); plt.show()
corr = scored.score.corr(scored.fwd)
print(f'Score vs forward-return correlation: {corr:+.4f}')
EDGE_OK = bool(by_sig.loc['BUY', 'mean_fwd'] > by_sig.loc['AVOID', 'mean_fwd'])
print('BUY beats AVOID on forward returns:', EDGE_OK)

## 2. Backtest
Decisions use the bar *close*; fills happen at the *next open*. Long-only, fully invested, fees and slippage on every fill.

In [ ]:
result = run_backtest(bars, SignalStrategy(), CONFIG)
m = result.metrics
summary = pd.Series({
    'Total return': f"{m['total_return']:.1%}", 'Buy & hold': f"{m['benchmark_return']:.1%}",
    'Excess vs B&H': f"{m['excess_return']:.1%}", 'CAGR': f"{m['cagr']:.1%}" if m['cagr'] is not None else 'n/a',
    'Max drawdown': f"{m['max_drawdown']:.1%}",
    'Sharpe': f"{m['sharpe']:.2f}" if m['sharpe'] is not None else 'n/a',
    'Trades': m['num_trades'], 'Win rate': f"{m['win_rate']:.0%}" if m['win_rate'] is not None else 'n/a',
    'Profit factor': f"{m['profit_factor']:.2f}" if m['profit_factor'] is not None else 'n/a',
    'Time in market': f"{m['exposure']:.0%}", 'Fees paid': f"${m['total_fees']:,.0f}",
    'Final equity': f"${m['final_equity']:,.0f}"}, name=SYMBOL)
summary.to_frame()

In [ ]:
eq = pd.DataFrame([asdict(p) for p in result.equity]).set_index('timestamp')
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
eq.equity.plot(ax=ax[0], label='Strategy'); eq.benchmark_equity.plot(ax=ax[0], label='Buy & hold', alpha=.7)
ax[0].fill_between(eq.index, eq.equity.min(), eq.equity.max(), where=eq.in_position, alpha=.08, color='g', label='In position')
ax[0].legend(); ax[0].set_title('Equity curve')
eq.drawdown.plot.area(ax=ax[1], color='r', alpha=.4); ax[1].set_title('Drawdown'); plt.tight_layout(); plt.show()

In [ ]:
trades = pd.DataFrame([asdict(t) for t in result.trades])
if trades.empty:
    print('No trades generated.')
else:
    display(trades.exit_reason.value_counts().to_frame('count'))
    trades.return_pct.hist(bins=30); plt.title('Per-trade return distribution'); plt.axvline(0, c='k'); plt.show()
    display(trades.sort_values('pnl').iloc[[0, 1, -2, -1]][['entry_time', 'exit_time', 'entry_price', 'exit_price', 'return_pct', 'bars_held', 'exit_reason']])

## 3. Sensitivity
A result that only works for one exact setting is probably overfit.

In [ ]:
rows = []
for fee in (0, 20, 40, 60, 100):
    r = run_backtest(bars, SignalStrategy(), BacktestConfig(fee_bps=fee, slippage_bps=5)).metrics
    rows.append({'test': 'fee_bps', 'value': fee, 'return': r['total_return'], 'maxDD': r['max_drawdown'], 'trades': r['num_trades']})
for stop in (None, 2, 3, 4):
    r = run_backtest(bars, SignalStrategy(), BacktestConfig(stop_atr_mult=stop)).metrics
    rows.append({'test': 'stop_atr', 'value': stop, 'return': r['total_return'], 'maxDD': r['max_drawdown'], 'trades': r['num_trades']})
for enter, exit_ in ((55, 45), (60, 40), (65, 35), (70, 30), (75, 25)):
    r = run_backtest(bars, ScoreStrategy(enter, exit_), CONFIG).metrics
    rows.append({'test': 'enter/exit', 'value': f'{enter}/{exit_}', 'return': r['total_return'], 'maxDD': r['max_drawdown'], 'trades': r['num_trades']})
sens = pd.DataFrame(rows); display(sens)

## 4. All pairs, persisted runs and live-pipeline health (Azure SQL only)

In [ ]:
if USE_DB:
    from cryptoquant.backtesting.runner import run_backtests
    res = run_backtests(SignalStrategy(), CONFIG, save=False)
    allp = pd.DataFrame([r for r in res if 'error' not in r])
    display(allp[['symbol', 'total_return', 'benchmark_return', 'excess_return', 'max_drawdown', 'sharpe', 'num_trades', 'win_rate']]
            .sort_values('excess_return', ascending=False))
    skipped = [r for r in res if 'error' in r]
    if skipped: display(pd.DataFrame(skipped))
else:
    print('Skipped (no database).')

In [ ]:
if USE_DB:
    from sqlalchemy import text
    display(pd.read_sql(text('SELECT TOP 20 * FROM crypto.vw_backtest_summary ORDER BY created_at DESC'), session.bind))
    live = pd.read_sql(text('''
        SELECT tp.symbol, MAX(ta.timestamp) AS latest_analysis,
               (SELECT TOP 1 signal FROM crypto.technical_analysis x
                 WHERE x.trading_pair_id = ta.trading_pair_id ORDER BY x.timestamp DESC) AS latest_signal,
               (SELECT TOP 1 technical_score FROM crypto.technical_analysis x
                 WHERE x.trading_pair_id = ta.trading_pair_id ORDER BY x.timestamp DESC) AS latest_score,
               SUM(CASE WHEN ta.signal IS NULL THEN 1 ELSE 0 END) AS unscored_rows
        FROM crypto.technical_analysis ta JOIN crypto.trading_pairs tp ON tp.id = ta.trading_pair_id
        GROUP BY tp.symbol, ta.trading_pair_id ORDER BY tp.symbol'''), session.bind)
    live['hours_stale'] = (pd.Timestamp.utcnow().tz_localize(None) - pd.to_datetime(live.latest_analysis)).dt.total_seconds() / 3600
    display(live)
else:
    print('Skipped (no database).')

## 5. Go / no-go checklist before the AI-intelligence phase
Fill in your own judgement below the computed values.

In [ ]:
checks = {
    'Scores populated (no NULL signals after warm-up)': bool(df.signal.notna().sum() > 0),
    'BUY forward return > AVOID forward return': EDGE_OK,
    'Strategy beats buy & hold (return) after costs': bool(m['excess_return'] > 0),
    'Strategy max drawdown shallower than buy & hold': bool(m['max_drawdown'] > (eq.benchmark_equity / eq.benchmark_equity.cummax() - 1).min()),
    'Enough trades for significance (>= 30)': bool(m['num_trades'] >= 30),
    'Still profitable at 100 bps fees': bool(sens[(sens.test == 'fee_bps') & (sens.value == 100)]['return'].iloc[0] > 0),
}
pd.Series(checks, name='pass').to_frame()

**Interpretation guide**
- *Edge fails* -> adjust `ScoringConfig` (thresholds/weights in `analytics/scoring.py`), run `python scripts/rescore_analysis.py`, re-run this notebook.
- *Edge passes but costs kill it* -> lower trading frequency (wider enter/exit gap, daily bars) before adding AI.
- *Most checks pass on several pairs* -> proceed to the strategy/AI phase.